## Export Models and Save Metrics to Google Drive

This section modifies the notebook to export your trained gesture recognition models directly to Google Drive and saves a text file containing the test performance metrics (loss and accuracy) for each model. This ensures persistence and easy access to your model artifacts and results.

**Note on Training and Validation Metrics**: The full training and validation history (metrics for each epoch) is displayed in the output of the model training cells. However, due to the structure of the `mediapipe-model-maker` API, these detailed metrics are not directly accessible as a programmatic object after the `create` method finishes. The generated text files will therefore focus on saving the final test loss and accuracy obtained from `model.evaluate()`.

Project: /mediapipe/_project.yaml
Book: /mediapipe/_book.yaml

<link rel="stylesheet" href="/mediapipe/site.css">

# Hand gesture recognition model customization guide

<table align="left" class="buttons">
  <td>
    <a href="https://colab.research.google.com/github/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/colab-logo-32px_1920.png" alt="Colab logo"> Run in Colab
    </a>
  </td>

  <td>
    <a href="https://github.com/googlesamples/mediapipe/blob/main/examples/customization/gesture_recognizer.ipynb" target="_blank">
      <img src="https://developers.google.com/static/mediapipe/solutions/customization/github-logo-32px_1920.png" alt="GitHub logo">
      View on GitHub
    </a>
  </td>
</table>

In [ ]:
#@title License information
# Copyright 2023 The MediaPipe Authors.
# Licensed under the Apache License, Version 2.0 (the "License");
#
# you may not use this file except in compliance with the License.
# You may obtain a copy of the License at
#
# https://www.apache.org/licenses/LICENSE-2.0
#
# Unless required by applicable law or agreed to in writing, software
# distributed under the License is distributed on an "AS IS" BASIS,
# WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
# See the License for the specific language governing permissions and
# limitations under the License.

The MediaPipe Model Maker package is a low-code solution for customizing on-device machine learning (ML) Models.

This notebook shows the end-to-end process of customizing a gesture recognizer model for recognizing some common hand gestures in the [HaGRID](https://www.kaggle.com/datasets/innominate817/hagrid-sample-30k-384p) dataset.

## Prerequisites

Install the MediaPipe Model Maker package.

In [ ]:
!pip install --upgrade pip
!pip install mediapipe-model-maker
#connect to 2025.7 runtime

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Import the required libraries.



In [ ]:
from google.colab import files
import os
import tensorflow as tf
assert tf.__version__.startswith('2')

from mediapipe_model_maker import gesture_recognizer

import matplotlib.pyplot as plt

## Simple End-to-End Example

This end-to-end example uses Model Maker to customize a model for on-device gesture recognition.

### Get the dataset

The dataset for gesture recognition in model maker requires the following format: `<dataset_path>/<label_name>/<img_name>.*`. In addition, one of the label names (`label_names`) must be `none`. The `none` label represents any gesture that isn't classified as one of the other gestures.

This example uses a rock paper scissors dataset sample which is downloaded from GCS.

In [ ]:
#Since we are doing an aboloation study, we will use 25 images per gesture instead of 100

In [ ]:
dataset_path = "/content/drive/MyDrive/Intro to deep learning/hagrid_25_images"

### Run the example
The workflow consists of 4 steps which have been separated into their own code blocks.

**Load the dataset**

Load the dataset located at `dataset_path` by using the `Dataset.from_folder` method. When loading the dataset, run the pre-packaged hand detection model from MediaPipe Hands to detect the hand landmarks from the images. Any images without detected hands are ommitted from the dataset. The resulting dataset will contain the extracted hand landmark positions from each image, rather than images themselves.

The `HandDataPreprocessingParams` class contains two configurable options for the data loading process:
* `shuffle`: A boolean controlling whether to shuffle the dataset. Defaults to true.
* `min_detection_confidence`: A float between 0 and 1 controlling the confidence threshold for hand detection.

Split the dataset: 80% for training, 10% for validation, and 10% for testing.

In [ ]:
data = gesture_recognizer.Dataset.from_folder(
    dirname=dataset_path,
    hparams=gesture_recognizer.HandDataPreprocessingParams()
)
train_data, rest_data = data.split(0.8)
validation_data, test_data = rest_data.split(0.5)

**Train the model**

Train the custom gesture recognizer by using the create method and passing in the training data, validation data, model options, and hyperparameters. For more information on model options and hyperparameters, see the [Hyperparameters](#hyperparameters) section below.

In [ ]:
hparams = gesture_recognizer.HParams(export_dir="exported_model", epochs=20)
options = gesture_recognizer.GestureRecognizerOptions(hparams=hparams)
model = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)


**Evaluate the model performance**

After training the model, evaluate it on a test dataset and print the loss and accuracy metrics.

In [ ]:
loss, acc = model.evaluate(test_data, batch_size=1)
print(f"Test loss:{loss}, Test accuracy:{acc}")

**Export to Tensorflow Lite Model**

After creating the model, convert and export it to a Tensorflow Lite model format for later use on an on-device application. The export also includes model metadata, which includes the label file.

In [ ]:
model.export_model()
!ls exported_model

In [ ]:
import os

# Define base paths
base_drive_path = "/content/drive/MyDrive/Intro to deep learning"
models_dir = os.path.join(base_drive_path, "models")
metrics_dir = os.path.join(base_drive_path, "metrics")

# Create directories if they don't exist
os.makedirs(models_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

# Define file names for the baseline model
model_name = "baseline_25imgs"
source_model_path = "/content/exported_model/gesture_recognizer.task"
destination_model_path = os.path.join(models_dir, f"{model_name}.task")
metrics_filepath = os.path.join(metrics_dir, f"{model_name}_metrics.txt")

# Copy and rename the model
# The source path for export_model is `exported_model/gesture_recognizer.task`
# and `model.export_model()` places it in `/content/exported_model/gesture_recognizer.task`.
# Using quotes for paths with spaces
!cp "{source_model_path}" "{destination_model_path}"

# Save metrics to a text file
# 'loss' and 'acc' are assumed to be available from the previous model.evaluate() call.
with open(metrics_filepath, "w") as f:
    f.write(f"Model: {model_name}\n")
    f.write(f"Test Loss: {loss}\n")
    f.write(f"Test Accuracy: {acc}\n")

print(f"Baseline model saved to: {destination_model_path}")
print(f"Baseline metrics saved to: {metrics_filepath}")

## Run the model on-device
DO NOT run this in colab.
Use this on actual device to run .task file

-Daniel

In [ ]:
# @title
#import sys
#print(sys.executable)
# Imports necessary modules.
import mediapipe as mp
import cv2
import os
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# --- IMPORTANT NOTE FOR LOCAL EXECUTION ---
# This code is designed to run in a local Python environment on your machine,
# not directly within this Google Colab notebook, as Colab generally cannot
# access local webcams. Save this code as a .py file and run it locally.
# Ensure 'gesture_recognizer.task' is in the same directory or update its path.
# ------------------------------------------

# Create a GestureRecognizer object.
# Ensure this path is correct relative to where you run the script locally.
model_path = os.path.abspath("C:/Users/Daniel Odi/Desktop/UCF/Spring26/DeepLearningProject/gesture_recognizer.task")

# Check if the model file exists
if not os.path.exists(model_path):
    print(f"Error: Model file not found at {model_path}. ")
    print("Please ensure"
    " is in the correct directory.")
else:
    print(f"Loading model from: {model_path}")
    base_options = python.BaseOptions(model_asset_path=model_path)
    options = vision.GestureRecognizerOptions(base_options=base_options)
    recognizer = vision.GestureRecognizer.create_from_options(options)

    # Initialize webcam capture
    cap = cv2.VideoCapture(0) # 0 indicates the default webcam

    if not cap.isOpened():
        print("Error: Could not open webcam. Please check if it's connected and not in use.")
    else:
        print("Webcam opened successfully. Press 'q' to quit.")
        while True:
            ret, frame = cap.read()
            if not ret:
                print("Error: Failed to grab frame.")
                break

            # Convert the BGR image from OpenCV to RGB, as MediaPipe typically expects RGB
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)

            # Create an mp.Image object from the RGB numpy array
            mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame_rgb)

            # Run gesture recognition.
            recognition_result = recognizer.recognize(mp_image)

            # Display the most likely gesture on the frame
            if recognition_result.gestures and recognition_result.gestures[0]:
                top_gesture = recognition_result.gestures[0][0]
                gesture_text = f"Gesture: {top_gesture.category_name} ({top_gesture.score:.2f})"
                cv2.putText(frame, gesture_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 255, 0), 2, cv2.LINE_AA)
            else:
                cv2.putText(frame, "No gesture detected", (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (0, 0, 255), 2, cv2.LINE_AA)

            # Display the frame
            cv2.imshow('Gesture Recognition', frame)

            # Break the loop when 'q' is pressed
            if cv2.waitKey(1) & 0xFF == ord('q'):
                break

        # Release the webcam and destroy all OpenCV windows
        cap.release()
        cv2.destroyAllWindows()


## Hyperparameters {:#hyperparameters}


This is where we will do our abolation study. We will change each hyper parameter 3 times and compare it to the baseline run with default hyperparameters.

You can further customize the model using the `GestureRecognizerOptions` class, which has two optional parameters for `ModelOptions` and `HParams`. Use the `ModelOptions` class to customize parameters related to the model itself, and the `HParams` class to customize other parameters related to training and saving the model.

`ModelOptions` has one customizable parameter that affects accuracy:
* `dropout_rate`: The fraction of the input units to drop. Used in dropout layer. Defaults to 0.05.
* `layer_widths`: A list of hidden layer widths for the gesture model. Each element in the list will create a new hidden layer with the specified width. The hidden layers are separated with BatchNorm, Dropout, and ReLU. Defaults to an empty list(no hidden layers).

`HParams` has the following list of customizable parameters which affect model accuracy:
* `learning_rate`: The learning rate to use for gradient descent training. Defaults to 0.001.
* `batch_size`: Batch size for training. Defaults to 2.
* `epochs`: Number of training iterations over the dataset. Defaults to 10.
* `steps_per_epoch`: An optional integer that indicates the number of training steps per epoch. If not set, the training pipeline calculates the default steps per epoch as the training dataset size divided by batch size.
* `shuffle`: True if the dataset is shuffled before training. Defaults to False.
* `lr_decay`: Learning rate decay to use for gradient descent training. Defaults to 0.99.
* `gamma`: Gamma parameter for focal loss. Defaults to 2

Additional `HParams` parameter that does not affect model accuracy:
* `export_dir`: The location of the model checkpoint files and exported model files.

For example, the following trains a new model with the dropout_rate of 0.2 and learning rate of 0.003.

In [ ]:
hparams = gesture_recognizer.HParams(learning_rate=0.003, export_dir="exported_model_2")
model_options = gesture_recognizer.ModelOptions(dropout_rate=0.2)
options = gesture_recognizer.GestureRecognizerOptions(model_options=model_options, hparams=hparams)
model_2 = gesture_recognizer.GestureRecognizer.create(
    train_data=train_data,
    validation_data=validation_data,
    options=options
)

Evaluate the newly trained model.

In [ ]:
loss, accuracy = model_2.evaluate(test_data)
print(f"Test loss:{loss}, Test accuracy:{accuracy}")

### Automate Hyperparameter Study

This section automates the training, evaluation, and saving of models for different hyperparameter combinations. It reads configurations from a CSV file, trains a model for each configuration, and saves the model and its test metrics to Google Drive.

In [ ]:
import pandas as pd
import os

# Define the path for the hyperparameters CSV file in Google Drive
hyperparameters_csv_path = os.path.join(base_drive_path, "hyperparameters.csv")

# Create a sample DataFrame for hyperparameters
# Each row represents a single change from the baseline
# Baseline values: dropout_rate=0.05, learning_rate=0.001, epochs=20

data = {
    'run_name': [
        'lr_0.0005',
        'lr_0.002',
        'dropout_0.1',
        'dropout_0.15',
        'epochs_10',
        'epochs_30'
    ],
    'learning_rate': [
        0.0005,  # Changed from baseline 0.001
        0.002,   # Changed from baseline 0.001
        0.001,   # Baseline
        0.001,   # Baseline
        0.001,   # Baseline
        0.001    # Baseline
    ],
    'dropout_rate': [
        0.05,    # Baseline
        0.05,    # Baseline
        0.1,     # Changed from baseline 0.05
        0.15,    # Changed from baseline 0.05
        0.05,    # Baseline
        0.05     # Baseline
    ],
    'epochs': [
        20,      # Baseline
        20,      # Baseline
        20,      # Baseline
        20,      # Baseline
        10,      # Changed from baseline 20
        30       # Changed from baseline 20
    ]
}
hp_df = pd.DataFrame(data)

# Save the DataFrame to a CSV file in Google Drive
hp_df.to_csv(hyperparameters_csv_path, index=False)

print(f"Sample hyperparameters CSV created at: {hyperparameters_csv_path}")
print("You can edit this file in Google Drive to define your desired hyperparameter combinations.")
print("Each row in the CSV will be used to train a new model.")

In [ ]:
# Load the hyperparameters from the CSV file
hyperparameters_to_test = pd.read_csv(hyperparameters_csv_path)

# Define base paths for saving models and metrics
base_drive_path = "/content/drive/MyDrive/Intro to deep learning"
models_dir = os.path.join(base_drive_path, "models")
metrics_dir = os.path.join(base_drive_path, "metrics")

# Ensure directories exist
os.makedirs(models_dir, exist_ok=True)
os.makedirs(metrics_dir, exist_ok=True)

print(f"Starting automated training for {len(hyperparameters_to_test)} configurations...")

for index, row in hyperparameters_to_test.iterrows():
    run_name = row['run_name']
    current_learning_rate = row['learning_rate']
    current_dropout_rate = row['dropout_rate']
    current_epochs = int(row['epochs']) # Ensure epochs is an integer

    print(f"\n--- Training model for configuration: {run_name} ---")
    print(f"  Learning Rate: {current_learning_rate}")
    print(f"  Dropout Rate: {current_dropout_rate}")
    print(f"  Epochs: {current_epochs}")

    # Configure HParams and ModelOptions for the current run
    current_hparams = gesture_recognizer.HParams(
        learning_rate=current_learning_rate,
        epochs=current_epochs,
        export_dir=f"exported_model_{run_name}" # Unique export directory for each model
    )
    current_model_options = gesture_recognizer.ModelOptions(
        dropout_rate=current_dropout_rate
    )
    current_options = gesture_recognizer.GestureRecognizerOptions(
        model_options=current_model_options,
        hparams=current_hparams
    )

    # Train the model
    model = gesture_recognizer.GestureRecognizer.create(
        train_data=train_data,
        validation_data=validation_data,
        options=current_options
    )

    # Evaluate the model
    test_loss, test_accuracy = model.evaluate(test_data, batch_size=1)
    print(f"  Test Loss: {test_loss:.4f}, Test Accuracy: {test_accuracy:.4f}")

    # Export the model
    model.export_model()

    # Define paths for saving the model and metrics to Google Drive
    source_model_path = os.path.join(f"exported_model_{run_name}", "gesture_recognizer.task")
    destination_model_path = os.path.join(models_dir, f"{run_name}.task")
    metrics_filepath = os.path.join(metrics_dir, f"{run_name}_metrics.txt")

    # Copy the model to Google Drive
    !cp "{source_model_path}" "{destination_model_path}"

    # Save metrics to a text file in Google Drive
    with open(metrics_filepath, "w") as f:
        f.write(f"Model: {run_name}\n")
        f.write(f"Learning Rate: {current_learning_rate}\n")
        f.write(f"Dropout Rate: {current_dropout_rate}\n")
        f.write(f"Epochs: {current_epochs}\n")
        f.write(f"Test Loss: {test_loss}\n")
        f.write(f"Test Accuracy: {test_accuracy}\n")

    print(f"  Model saved to: {destination_model_path}")
    print(f"  Metrics saved to: {metrics_filepath}")

print("\nAutomated training complete!")

### Summarize Performance Results

This section reads the saved metric files from Google Drive and displays the performance (test loss and accuracy) for each hyperparameter configuration in a table, allowing for easy comparison.

In [ ]:
import pandas as pd
import os

# Define the path to the metrics directory in Google Drive
metrics_dir = os.path.join("/content/drive/MyDrive/Intro to deep learning", "metrics")

# List to store extracted metrics
all_metrics = []

# Iterate over each file in the metrics directory
for filename in os.listdir(metrics_dir):
    if filename.endswith("_metrics.txt"):
        filepath = os.path.join(metrics_dir, filename)
        metrics = {}
        with open(filepath, "r") as f:
            for line in f:
                line = line.strip()
                if ":" in line:
                    key, value = line.split(":", 1)
                    metrics[key.strip()] = value.strip()
        all_metrics.append(metrics)

# Create a DataFrame from the collected metrics
metrics_df = pd.DataFrame(all_metrics)

# Ensure 'Test Loss' and 'Test Accuracy' are numeric for proper comparison
metrics_df['Test Loss'] = pd.to_numeric(metrics_df['Test Loss'])
metrics_df['Test Accuracy'] = pd.to_numeric(metrics_df['Test Accuracy'])

# Sort by run name for consistent display
metrics_df = metrics_df.sort_values(by='Model').reset_index(drop=True)

display(metrics_df)